# 02 — Covariate shift: shift-corrected deep local-covariance regions
Parkinson's telemonitoring, bivariate response (motor UPDRS, total UPDRS).

**Shift design** (Gan & Liu 2025, Sec. 5): 1000 training points; the test points are drawn from the
remaining 4875 records by importance sampling with weight w(X) = exp[k {s(X1)+s(X2)+s(X3)+s(X4)}],
s(.) = min–max scaling, k = 1, …, 7 (heavier shift as k grows), 90% nominal coverage, B = 1000.

**Methods**

| Method | Shift correction | Source |
|---|---|---|
| SCP-L2-LR | none | Gan & Liu comparison method |
| WCP-L2-LR | weighted CP | Tibshirani et al. (2019), compared in Gan & Liu |
| EU | none | Gan & Liu Eq. (4) |
| EUc-LR | resampling | Gan & Liu Algorithm 2 (Sec. 3.1 version, see below) |
| Deep-LocalCov | none | Braun et al. (2026) |
| Deep-LocalCov + WCP | weighted CP | **ours**: local Mahalanobis score + Tibshirani weighting |
| Deep-LocalCov + EUc | resampling | **ours**: Gan & Liu Eq. (8) on whitened residuals + their resampling |

**Ablations**: EUc-LR with Algorithm 2 read literally; both of our methods with the true (oracle)
density ratio instead of the estimated one.
(The oracle ratio is exact for sampling with replacement; with the default distinct-record sampling it is approximate.)

**Resampling direction.** Gan & Liu Sec. 3.1 and the proof of their Theorem 3 draw the pseudo-training
data from the training distribution and the pseudo-test point from the reweighted distribution;
their Algorithm 2 prints the opposite. The main method follows Sec. 3.1 + proof; the literal
Algorithm 2 is run as an ablation.

Settings marked **[paper]** come from the papers, **[my choice]** are not specified there.

In [ ]:
import os, sys, math, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath("../src"))
import cpshift as cp
import deep

print("device:", deep.DEVICE)

## Configuration

In [ ]:
QUICK = False   # True = fast smoke test

CFG = dict(
    SEED=2026,               # [my choice]
    ALPHA=0.10,              # [paper] 90% nominal coverage
    N_TRAIN=1000,            # [paper] Gan & Liu Sec. 5
    N_TEST=1000,             # [paper] Gan & Liu Sec. 5: 1000 test points drawn by importance sampling
    K_VALUES=[1, 2, 3, 4, 5, 6, 7],   # [paper] shift intensity
    N_REPS=20,               # [my choice] number of random splits (not stated in Gan & Liu)
    B=1000,                  # [paper] resamples
    SAMPLE_REPLACE=False,    # [my reading] 1000 distinct test records (see ShiftSampler)
    SCP_FIT_FRAC=0.5,        # [my choice]
    # deep model: identical to notebook 01
    DEEP_CAL_FRAC=0.5, DEEP_VAL_FRAC=0.2, HIDDEN=64, N_LAYERS=2, LR=1e-3, WEIGHT_DECAY=1e-4,
    BATCH=64, MAX_EPOCHS=500, PATIENCE=30, EPS=1e-3,
)
CC = dict(WSC_DELTA=0.10, WSC_N_DIR=1000, WSC_GRID=50, CEC_J=10)   # [my choice], as notebook 01
if QUICK:
    CFG.update(N_REPS=1, B=200, MAX_EPOCHS=40, K_VALUES=[1, 4, 7])
    CC.update(WSC_N_DIR=100)

OUT_DIR = "../results/covariate_shift"
os.makedirs(OUT_DIR, exist_ok=True)
CFG, CC

## Data

In [ ]:
df, X_all, Y_all = cp.load_parkinsons("../data/parkinsons_updrs.data")
print("X:", X_all.shape, "Y:", Y_all.shape)
sampler = cp.ShiftSampler(X_all)

## Diagnostic: how concentrated is the shifted test distribution?
ESS = effective number of pool records carrying the importance weight (out of 4875).
`unique_with_replacement` = distinct records among 1000 draws if sampled with replacement.

In [ ]:
_rng = np.random.default_rng(0)
_pool = _rng.permutation(len(Y_all))[CFG['N_TRAIN']:]
diag = pd.DataFrame([sampler.diagnostics(X_all, _pool, k, CFG['N_TEST'], _rng) for k in range(0, 8)]).round(1)
diag.to_csv(f"{OUT_DIR}/shift_diagnostics.csv", index=False)
diag

## Run: one training split per rep, all shift intensities k on that split

In [ ]:
MAIN = ['SCP-L2-LR', 'WCP-L2-LR', 'EU', 'EUc-LR',
        'Deep-LocalCov', 'Deep-LocalCov + WCP (ours)', 'Deep-LocalCov + EUc (ours)']
ABL = ['EUc-LR, Algorithm 2 literal', 'Deep-LocalCov + WCP, oracle r', 'Deep-LocalCov + EUc, oracle r']

rows, keep = [], {}
a, N = CFG['ALPHA'], len(Y_all)
t0 = time.time()
for rep in range(CFG['N_REPS']):
    rng = np.random.default_rng(CFG['SEED'] + rep)
    perm = rng.permutation(N)
    tr, pool = perm[:CFG['N_TRAIN']], perm[CFG['N_TRAIN']:]
    Xtr, Ytr = X_all[tr], Y_all[tr]

    # parts that do not depend on k
    scp = cp.SplitL2LR(Xtr, Ytr, rng, CFG['SCP_FIT_FRAC'])
    eu = cp.EUJackknife(Xtr, Ytr, a)
    net = deep.DeepLocalCov(Xtr, Ytr, CFG, rng, CFG['SEED'] + rep)
    Xc = Xtr[net.cal_idx]
    cal = net.outputs(Xc, Ytr[net.cal_idx])
    centers = cp.kmeanspp(cp.standardize_from(Xtr, Xtr), CC['CEC_J'], rng)

    for k in CFG['K_VALUES']:
        te = sampler.sample(X_all, pool, k, CFG['N_TEST'], rng, replace=CFG['SAMPLE_REPLACE'])
        Xte, Yte = X_all[te], Y_all[te]
        test = net.outputs(Xte, Yte)
        r_hat = cp.fit_density_ratio(Xtr, Xte)          # estimated from unlabeled test inputs
        r_or = sampler.oracle_ratio(Xtr, k)              # true ratio (known by construction)

        res = {
            'SCP-L2-LR': scp.unweighted(Xte, Yte, a),
            'WCP-L2-LR': scp.weighted(Xte, Yte, a, r_hat),
            'EU': eu.predict(Xte, Yte),
            'EUc-LR': cp.EUcLR(Xtr, Ytr, r_hat, a, CFG['B'], rng, 'text').predict(Xte, Yte),
            'Deep-LocalCov': cp.deep_split(cal, test, a, net.ys),
            'Deep-LocalCov + WCP (ours)': cp.deep_weighted(cal, test, a, net.ys, r_hat(Xc), r_hat(Xte)),
            'Deep-LocalCov + EUc (ours)': cp.deep_euc(cal, test, a, net.ys, r_hat(Xc), CFG['B'], rng, 'text'),
            'EUc-LR, Algorithm 2 literal': cp.EUcLR(Xtr, Ytr, r_hat, a, CFG['B'], rng, 'algorithm').predict(Xte, Yte),
            'Deep-LocalCov + WCP, oracle r': cp.deep_weighted(cal, test, a, net.ys, r_or(Xc), r_or(Xte)),
            'Deep-LocalCov + EUc, oracle r': cp.deep_euc(cal, test, a, net.ys, r_or(Xc), CFG['B'], rng, 'text'),
        }
        names = MAIN + ABL
        Xs = cp.standardize_from(Xtr, Xte)
        w = cp.wsc(Xs, [res[m]['covered'].astype(float) for m in names],
                   cp.random_directions(CC['WSC_N_DIR'], Xte.shape[1], rng), CC['WSC_DELTA'], CC['WSC_GRID'])
        lab = cp.assign(Xs, centers)
        for m, wv in zip(names, w):
            v = res[m]['vol']
            fin = np.isfinite(v)
            rows.append(dict(rep=rep, k=k, method=m, group='main' if m in MAIN else 'ablation',
                             coverage=100 * res[m]['covered'].mean(),
                             volume_median=np.median(v),
                             volume_mean_finite=v[fin].mean() if fin.any() else np.nan,
                             pct_infinite=100 * (1 - fin.mean()),
                             WSC=100 * wv,
                             CEC_X=1e4 * cp.cec_x(lab, res[m]['covered'].astype(float), CC['CEC_J'], a)))
        if rep == 0:
            keep[k] = dict(res=res, Xte=Xte, Yte=Yte)
    print(f"rep {rep + 1}/{CFG['N_REPS']} done | deep epochs={len(net.hist)} | elapsed {time.time() - t0:.0f}s")

res_df = pd.DataFrame(rows)
res_df.to_csv(f"{OUT_DIR}/results_all_reps.csv", index=False)

## Table: coverage, volume and WSC by shift intensity k
Volume = median region area over test points (UPDRS²), averaged over reps
(median because weighted CP can give an infinite region for extreme test points).

In [ ]:
summ = (res_df.groupby(['method', 'k'])
        .agg(coverage=('coverage', 'mean'), coverage_sd=('coverage', 'std'),
             volume=('volume_median', 'mean'), volume_sd=('volume_median', 'std'),
             pct_infinite=('pct_infinite', 'mean'), WSC=('WSC', 'mean'), WSC_sd=('WSC', 'std'),
             CEC_X=('CEC_X', 'mean'))
        .round(2).reset_index())
summ.to_csv(f"{OUT_DIR}/summary_by_k.csv", index=False)

order = MAIN + ABL
for col in ['coverage', 'volume', 'WSC']:
    tab = summ.pivot(index='method', columns='k', values=col).reindex(order)
    tab.to_csv(f"{OUT_DIR}/table_{col}_by_k.csv")
    print(f"\n{col.upper()} by k\n", tab.to_string())

overall = (res_df.groupby('method')
           .agg(coverage=('coverage', 'mean'), volume=('volume_median', 'mean'),
                pct_infinite=('pct_infinite', 'mean'), WSC=('WSC', 'mean'), CEC_X=('CEC_X', 'mean'))
           .reindex(order).round(2))
overall.to_csv(f"{OUT_DIR}/table_overall.csv")
print("\nAVERAGED OVER k\n", overall.to_string())

## Figure: coverage, volume and WSC versus shift intensity (main methods)

In [ ]:
style = {'SCP-L2-LR': ('tab:gray', 's', '--'), 'WCP-L2-LR': ('tab:brown', 's', '-'),
         'EU': ('tab:pink', 'o', '--'), 'EUc-LR': ('tab:red', 'o', '-'),
         'Deep-LocalCov': ('tab:cyan', '^', '--'),
         'Deep-LocalCov + WCP (ours)': ('tab:green', '^', '-'),
         'Deep-LocalCov + EUc (ours)': ('tab:blue', 'D', '-')}
fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))
for m in MAIN:
    s = summ[summ.method == m]
    c, mk, ls = style[m]
    ax[0].plot(s.k, s.coverage, color=c, marker=mk, ls=ls, label=m)
    ax[1].plot(s.k, s.volume, color=c, marker=mk, ls=ls)
    ax[2].plot(s.k, s.WSC, color=c, marker=mk, ls=ls)
ax[0].axhline(100 * (1 - a), color='k', lw=1, ls=':')
ax[0].set_ylabel('coverage (%)'); ax[1].set_ylabel('median region area (UPDRS$^2$)'); ax[2].set_ylabel('WSC (%)')
ax[1].set_yscale('log')
for x in ax:
    x.set_xlabel('shift intensity $k$')
fig.legend(loc='lower center', ncol=4, fontsize=8, frameon=False, bbox_to_anchor=(0.5, -0.08))
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fig_shift_curves.png", dpi=200, bbox_inches='tight'); plt.show()

## Paired comparison over reps (averaged over k)

In [ ]:
from scipy import stats
piv = res_df.groupby(['rep', 'method'])[['coverage', 'volume_median', 'WSC']].mean().unstack('method')
pairs = [('Deep-LocalCov + EUc (ours)', 'EUc-LR'), ('Deep-LocalCov + EUc (ours)', 'Deep-LocalCov'),
         ('Deep-LocalCov + WCP (ours)', 'WCP-L2-LR'), ('Deep-LocalCov + EUc (ours)', 'Deep-LocalCov + WCP (ours)')]
out = []
for m1, m2 in pairs:
    for met in ['coverage', 'volume_median', 'WSC']:
        d = (piv[met][m1] - piv[met][m2]).values
        out.append(dict(comparison=f"{m1} - {m2}", metric=met, mean_diff=d.mean(), sd=d.std(ddof=1),
                        p_t=stats.ttest_1samp(d, 0).pvalue if len(d) > 1 else np.nan,
                        p_wilcoxon=stats.wilcoxon(d).pvalue if len(d) > 5 else np.nan))
paired = pd.DataFrame(out).round(4)
paired.to_csv(f"{OUT_DIR}/paired_tests.csv", index=False)
paired

## Saved files

In [ ]:
print("Files saved in", os.path.abspath(OUT_DIR), ":", sorted(os.listdir(OUT_DIR)))
print("CONFIG:", CFG, "\nCC:", CC)